# Profitable Wallet Analysis

Loads processed trades from `data/trades_processed.parquet` (top-5% wallets by training-period P&L).
Wallet selection metrics are computed on **training data only** (`is_train=True`).
Final PnL plots cover the full dataset (training + test).

In [180]:
import math
from pathlib import Path

import pandas as pd
import numpy as np

## Parameters

In [181]:
import datetime

END_DATE_TRAIN = datetime.date(2026, 2, 1)

TRADES_PARQUET = Path("../data/trades_processed.parquet")

## 1 · Load processed trades

In [182]:
df = pd.read_parquet(TRADES_PARQUET)

# ensure correct dtypes after round-trip
df["dt"]               = pd.to_datetime(df["dt"], utc=True)
df["trade_value_usdc"] = df["trade_value_usdc"].astype(float)
df["final_value_usdc"] = df["final_value_usdc"].astype(float)
df["size"]             = df["size"].astype(float)
df["price"]            = df["price"].astype(float)

# ── PnL per fill ──────────────────────────────────────────────────────────────
df["pnl"] = np.where(
    df["side"] == "BUY",
    df["final_value_usdc"] - df["trade_value_usdc"],
    # SELL: we sold token at price p, implicitly holding the opposite token
    # pnl = trade_value - final_value
    df["trade_value_usdc"] - df["final_value_usdc"],
)

# ── notional per fill ─────────────────────────────────────────────────────────
df["notional"] = np.where(
    df["side"] == "BUY",
    df["trade_value_usdc"],
    df["size"] * (1 - df["price"]),
)

print(f"Loaded {len(df):,} trade records for {df['wallet'].nunique():,} wallets.")
print(f"  training rows : {df['is_train'].sum():,}")
print(f"  test rows     : {(~df['is_train']).sum():,}")
df.head(3)

Loaded 5,470,425 trade records for 15,538 wallets.
  training rows : 3,728,856
  test rows     : 1,741,569


,wallet,condition_id,dt,side,asset,outcome,outcomeIndex,size,price,trade_value_usdc,final_value_usdc,token_winner,final_price,name,pseudonym,transactionHash,title,is_train,pnl,notional
0,0x1d3fd83676aabe718c13cba008e0a774b2127fec,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,2026-02-01 00:27:12+00:00,BUY,2464494260851836388524244660612053848040667625...,No,1,36.26,0.988447,35.8411,36.26,True,1.0,Midrunner,Woozy-Copy,0xbe77f57ad7cf076d224c718458709a7edc61b1e40a7f...,Will Jonathan Micallef win by KO or TKO?,True,0.4189,35.8411
1,0x5ee1c416fbe1c3e6a8bc51b74dd816c43276e116,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,2026-02-01 00:39:02+00:00,BUY,2464494260851836388524244660612053848040667625...,No,1,1287.00,0.999000,1285.7130,1287.00,True,1.0,WuTangClan,Tempting-Catalogue,0x4c82ec38eac8594af4ca7d8cedc668bf64796f2920bf...,Will Jonathan Micallef win by KO or TKO?,True,1.2870,1285.7130
2,0x5ee1c416fbe1c3e6a8bc51b74dd816c43276e116,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,2026-02-01 00:40:28+00:00,BUY,2464494260851836388524244660612053848040667625...,No,1,706.00,0.999000,705.2940,706.00,True,1.0,WuTangClan,Tempting-Catalogue,0x05840ca9cf2976af584be395b84973e3bf2577585dd8...,Will Jonathan Micallef win by KO or TKO?,True,0.7060,705.2940


In [183]:
len(df[df['is_train'] == False])

1741569

In [184]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
df[df["is_train"]].groupby("wallet").agg(
    num_trades  = ("transactionHash", "count"),
    num_markets = ("condition_id",    "nunique"),
    pnl_usdc    = ("pnl",             "sum"),
).describe(percentiles=QUANTILES)

,num_trades,num_markets,pnl_usdc
count,15538.000000,15538.000000,1.553800e+04
mean,239.983009,50.326876,3.156339e+03
std,4437.073859,283.127688,2.720692e+04
min,1.000000,1.000000,-1.663874e+04
0.1%,1.000000,1.000000,1.790160e+02
1%,1.000000,1.000000,1.825421e+02
5%,1.000000,1.000000,1.978964e+02
10%,1.000000,1.000000,2.193000e+02
25%,2.000000,2.000000,3.188124e+02
50%,7.000000,5.000000,6.508811e+02


## 2 · Weighted-pnl volatility per wallet (training data only)

Step 1 — collapse fills into **timestamp buckets** `(wallet, dt)` using **training rows only**.  
Step 2 — apply the weighted-return volatility formula across those buckets.

In [185]:
import math
import numpy as np
import pandas as pd

def scaled_weighted_pnl_volatility(buckets: pd.DataFrame) -> float:
    """
    Compute capital-weighted PnL volatility scaled by sqrt(total PnL).

    Each row of `buckets` must contain:
        - notional : total capital in the bucket
        - pnl      : realized PnL in the bucket
    """
    if len(buckets) < 2:
        return float("nan")

    w = buckets["notional"].to_numpy()
    pnl = buckets["pnl"].to_numpy()

    total_w = w.sum()
    total_pnl = pnl.sum()

    if total_w == 0 or total_pnl <= 0:
        return float("nan")

    # weighted mean PnL
    mean = np.sum(w * pnl) / total_w

    # weighted variance
    variance = np.sum(w * (pnl - mean) ** 2) / total_w
    sigma = math.sqrt(variance)

    # scale by sqrt of total PnL
    sigma_scaled = sigma / math.sqrt(total_pnl)

    return sigma_scaled


def _wallet_metrics_from_buckets(group: pd.DataFrame) -> pd.Series:
    """Compute per-wallet metrics from a pre-aggregated bucket DataFrame."""
    pnl = group["pnl"].to_numpy()
    total_notional = group["notional"].sum()
    total_pnl = pnl.sum()

    if total_pnl <= 0:
        top5_pnl_pct = float("nan")
        top_market_pnl_pct = float("nan")
    else:
        top5_pnl = np.sort(pnl)[-5:].sum()
        top5_pnl_pct = top5_pnl / total_pnl
        top_market_pnl_pct = group.groupby("condition_id")["pnl"].sum().max() / total_pnl

    return pd.Series({
        "pnl_volatility":     scaled_weighted_pnl_volatility(group),
        "num_buckets":        len(group),
        "num_markets":        len(group["condition_id"].unique()),
        "total_notional":     total_notional,
        "total_pnl":          total_pnl,
        "top5_pnl_pct":       top5_pnl_pct,
        "top_market_pnl_pct": top_market_pnl_pct,
    })


def compute_wallet_metrics(df_slice: pd.DataFrame) -> pd.DataFrame:
    """
    Given a slice of the fills DataFrame (with columns: wallet, dt, condition_id,
    notional, pnl), aggregate into 5-minute buckets and compute per-wallet metrics.

    Returns a DataFrame indexed by wallet with columns:
        pnl_volatility, num_buckets, num_markets, total_notional,
        total_pnl, top5_pnl_pct, top_market_pnl_pct, return
    """
    tmp = df_slice.copy()
    tmp["dt_floored"] = tmp["dt"].dt.floor("5min")

    buckets = (
        tmp.groupby(["wallet", "dt_floored", "condition_id"], sort=False)
        .agg(notional=("notional", "sum"), pnl=("pnl", "sum"))
        .reset_index()
    )
    buckets = buckets[buckets["notional"] > 0].copy()

    result = (
        buckets.groupby("wallet", sort=False)
        .apply(_wallet_metrics_from_buckets, include_groups=False)
        .reset_index()
    )
    result["return"] = result["total_pnl"] / result["total_notional"]
    return result, buckets

In [186]:
# ── compute metrics on training slice ────────────────────────────────────────
df_train = df[df["is_train"]]
wallet_vol_train, buckets_train = compute_wallet_metrics(df_train)
wallet_vol_train = wallet_vol_train.sort_values("pnl_volatility").reset_index(drop=True)

print(f"Training timestamp buckets: {len(buckets_train):,}  across {buckets_train['wallet'].nunique():,} wallets.")
buckets_train.head(5)

Training timestamp buckets: 2,298,822  across 15,538 wallets.


,wallet,dt_floored,condition_id,notional,pnl
0,0x1d3fd83676aabe718c13cba008e0a774b2127fec,2026-02-01 00:25:00+00:00,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,35.841100,0.418900
1,0x5ee1c416fbe1c3e6a8bc51b74dd816c43276e116,2026-02-01 00:35:00+00:00,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,1285.713000,1.287000
2,0x5ee1c416fbe1c3e6a8bc51b74dd816c43276e116,2026-02-01 00:40:00+00:00,0x00005c113846891686d99ff1d145ded8bdccfa028df0...,1557.441000,1.559000
3,0x1fc793e8b69e49d77eece6b599756dd1d93555f6,2026-01-16 07:30:00+00:00,0x000196d079896702aec8581397e8339a08a087ba31aa...,1.399994,0.227906
4,0x204f72f35326db932158cba6adff0b9a1da95e14,2026-01-16 07:30:00+00:00,0x000196d079896702aec8581397e8339a08a087ba31aa...,100.300000,17.700000


In [206]:
WALLET = '0xcceb22d524e186153cffe79f13c0aeb75889f030'

print("Training PnL: " + str(df_train[df_train['wallet'] == WALLET]['pnl'].sum()))

(buckets_train[buckets_train["wallet"] == WALLET]
 .groupby("wallet")
 .apply(scaled_weighted_pnl_volatility, include_groups=False)
 .head(5))

Training PnL: 929.9177199999997


wallet
0xcceb22d524e186153cffe79f13c0aeb75889f030    0.768612
dtype: float64

In [216]:
df[
    # (df["wallet"] == WALLET) & 
    (df['condition_id'] == '0x680784812145063cecedfa7f000654e89317e2aca612182884d7742ac6f801f9')
    ]

,wallet,condition_id,dt,side,asset,outcome,outcomeIndex,size,price,trade_value_usdc,final_value_usdc,token_winner,final_price,name,pseudonym,transactionHash,title,is_train,pnl,notional
2235793,0x01ada2b4158f9c5fda27cc56fe3bfe05b0ef9e91,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-22 22:15:42+00:00,BUY,7617928500154880183205143787090187101296671796...,No,1,37.090908,0.539216,19.999999,37.090908,True,1.0,0x01adA2b4158f9C5fdA27cC56fe3BfE05B0ef9e91-176...,Weird-Due,0x9177194093931d6a2d45c15b850dc0f6d15eba11467e...,Will Microsoft (MSFT) close above $450 end of ...,True,17.090909,19.999999
2235794,0x0b3492ec87748d0f32427e8be4cbc650ff67ffc3,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-07 21:53:59+00:00,SELL,7617928500154880183205143787090187101296671796...,No,1,6.660000,0.130000,0.865800,6.660000,True,1.0,16r0ob,Joyful-Monsoon,0x8f0ca917af7eaafbfd6000c56f2950f23b03314f9c4e...,Will Microsoft (MSFT) close above $450 end of ...,True,-5.794200,5.794200
2235795,0x0b3492ec87748d0f32427e8be4cbc650ff67ffc3,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-09 10:14:02+00:00,SELL,7617928500154880183205143787090187101296671796...,No,1,5.260000,0.190000,0.999400,5.260000,True,1.0,16r0ob,Joyful-Monsoon,0x7048ba4787852a98b463e86f7c0ee6af69f8abe9ce56...,Will Microsoft (MSFT) close above $450 end of ...,True,-4.260600,4.260600
2235796,0x0b3492ec87748d0f32427e8be4cbc650ff67ffc3,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-12 09:53:38+00:00,SELL,7617928500154880183205143787090187101296671796...,No,1,7.690000,0.120000,0.922800,7.690000,True,1.0,16r0ob,Joyful-Monsoon,0xccf9e6aedc10bc80595dc74fcc475405d0c8a41e3d3b...,Will Microsoft (MSFT) close above $450 end of ...,True,-6.767200,6.767200
2235797,0x0b3492ec87748d0f32427e8be4cbc650ff67ffc3,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-26 16:01:35+00:00,SELL,7617928500154880183205143787090187101296671796...,No,1,5.330000,0.230000,1.225900,5.330000,True,1.0,16r0ob,Joyful-Monsoon,0x3d71d86d86852b1416dd0591042b232285cdd44ad09e...,Will Microsoft (MSFT) close above $450 end of ...,True,-4.104100,4.104100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235907,0xcceb22d524e186153cffe79f13c0aeb75889f030,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-09 10:07:36+00:00,SELL,6041099785595455628610604883170151776302742194...,Yes,0,70.000000,0.810000,56.700000,0.000000,False,0.0,ScottsRoad,Energetic-Gasp,0x4a751d4a3f3ebadd2640dd675f0a58a3ad904cad8c2c...,Will Microsoft (MSFT) close above $450 end of ...,True,56.700000,13.300000
2235908,0xcceb22d524e186153cffe79f13c0aeb75889f030,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-13 19:23:08+00:00,SELL,6041099785595455628610604883170151776302742194...,Yes,0,80.000000,0.740000,59.200000,0.000000,False,0.0,ScottsRoad,Energetic-Gasp,0xa0f556ba5dac7a70ab345808ffe283c2c9bc95129086...,Will Microsoft (MSFT) close above $450 end of ...,True,59.200000,20.800000
2235909,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-16 05:43:54+00:00,BUY,7617928500154880183205143787090187101296671796...,No,1,24.500000,0.260000,6.370000,24.500000,True,1.0,PX5oo,Agreeable-Airbus,0x38558ad2da1c4701bb467ba6f04c68c38a5fdc961759...,Will Microsoft (MSFT) close above $450 end of ...,True,18.130000,6.370000
2235910,0xdabd4df1cc941e552b44f0c2e5f929b4851e6ada,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-20 14:37:40+00:00,BUY,6041099785595455628610604883170151776302742194...,Yes,0,13.000000,0.560000,7.280000,0.000000,False,0.0,danielsvoboda,Loyal-Enjoyment,0xd17177d593e94c77397d6d70b90799ee9e7ecaefc662...,Will Microsoft (MSFT) close above $450 end of ...,True,-7.280000,7.280000


In [188]:
# ── Step 2: volatility per wallet (training data) ─────────────────────────────
# wallet_vol_train is already computed above via compute_wallet_metrics()
wallet_vol = wallet_vol_train  # alias kept for downstream cells

print(f"Wallets with volatility: {wallet_vol['pnl_volatility'].notna().sum():,}")
wallet_vol.head(10)

Wallets with volatility: 12,603


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
0,0x85c097cbd8beb9744be599b44cb4bc2328ed22e8,0.004244,2.0,2.0,1746.489998,1967.970144,1.000000,0.500096,1.126814
1,0x9803b7b4d2c33103cc0e3a57f1fb664a39447850,0.005675,2.0,1.0,2021.109996,718.110946,1.000000,1.000000,0.355305
2,0x6f9bdd59ab2d8816d76e99faaa46b789d6154193,0.013724,2.0,2.0,992.760000,2003.240000,1.000000,0.500309,2.017849
3,0x3e588ff146eb4daea133af7bc2c14399e84d80c1,0.014752,2.0,2.0,1320.810000,1124.190000,1.000000,0.500440,0.851137
4,0xa3a417e434923aeaf97afe9a61b10ff9364e5106,0.014983,5353.0,84.0,83993.605200,1038.434800,0.015889,0.136524,0.012363
5,0xee4d91b7d0fad963c2a4bf7b53b6bfb0990ecae5,0.022542,2.0,2.0,441.741800,313.278200,1.000000,0.501274,0.709188
6,0x15bd8329499e9df59d863dfddb823228ebcc1b0e,0.024192,2.0,2.0,161.120400,198.119600,1.000000,0.501788,1.229637
7,0x61002f423f644853d492e65c8b77e91098a42120,0.029922,2.0,2.0,1182.747200,1222.612800,1.000000,0.500862,1.033706
8,0xdad8c74fa7577320b78cb9787411dd1a409cd128,0.040123,1572.0,43.0,82472.677480,1187.602520,0.030894,0.207550,0.014400
9,0x99172a37f94079d92c5ca6fe452656c069c87740,0.050611,2.0,2.0,903.089997,502.794266,1.000000,0.502275,0.556749


In [207]:
wallet_vol[wallet_vol["wallet"] == WALLET]

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
428,0xcceb22d524e186153cffe79f13c0aeb75889f030,0.768612,613.0,364.0,15936.98113,929.91772,0.384273,0.196102,0.05835


## 3 · Volatility distribution (training)

In [190]:
QUANTILES = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
N = wallet_vol["pnl_volatility"].notna().sum()

vol_stats = (
    wallet_vol["pnl_volatility"]
    .dropna()
    .quantile(QUANTILES)
    .rename_axis("quantile")
    .to_frame()
)
vol_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
vol_stats.loc["mean"] = [float("nan"), wallet_vol["pnl_volatility"].mean()]
vol_stats.loc["std"]  = [float("nan"), wallet_vol["pnl_volatility"].std()]
vol_stats["wallet_count"] = vol_stats["wallet_count"].astype("Int64")
vol_stats["pnl_volatility"] = vol_stats["pnl_volatility"].round(4)

vol_stats

,wallet_count,pnl_volatility
quantile,,
0.01,126,0.3013
0.05,630,0.9707
0.1,1260,1.5542
0.25,3151,3.0786
0.5,6302,6.4406
0.75,9452,14.4249
0.9,11343,28.5283
0.95,11973,39.9552
0.99,12477,92.2177


In [191]:
len(wallet_vol[(wallet_vol['num_buckets'] >= 20) &
               (wallet_vol['top5_pnl_pct'] <= 0.6)
               & (wallet_vol['return'] >= 0.1)
               ])

199

In [192]:
# ── Select best wallets based on training-period metrics ──────────────────────
filtered_wallets = wallet_vol[
    (wallet_vol['num_buckets'] >= 20) &
    # (wallet_vol['num_buckets'] <= 300) &
    (wallet_vol['top5_pnl_pct'] <= 0.4) &
    (wallet_vol['top_market_pnl_pct'] <= 0.5) &
    (wallet_vol['return'] >= 0.05) &
    (wallet_vol['pnl_volatility'] <= 0.8)
].sort_values("total_pnl", ascending=False)

print(f"Best wallets (selected on training data): {len(filtered_wallets):}")
filtered_wallets.head(20)

Best wallets (selected on training data): 24


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
397,0xcf0aca0d7a395202aec661c3666be9cc098e320a,0.727513,586.0,245.0,39565.522536,5946.969456,0.265758,0.078130,0.150307
411,0x02319ffcf7214a5ca8e972d409320c935ea545d0,0.746335,421.0,123.0,29709.337690,3276.802310,0.187134,0.151404,0.110295
350,0x29f8da02e78a87b2d9a844124ff570d5cee34d7b,0.664580,232.0,84.0,5816.015926,1213.979017,0.353252,0.192214,0.208730
253,0x2911e7b42a848d0a539ba88ac4a4e7d49480fed6,0.543717,114.0,71.0,8498.980125,1043.505849,0.207839,0.073912,0.122780
428,0xcceb22d524e186153cffe79f13c0aeb75889f030,0.768612,613.0,364.0,15936.981130,929.917720,0.384273,0.196102,0.058350
304,0x8f3ad9fea07b12c950039a668425083ea3d89bdb,0.607948,335.0,315.0,2956.189700,685.410300,0.358967,0.202973,0.231856
152,0x766f37ff68fb7e85b122ceb8ab3b1af67d6a9e74,0.381731,135.0,21.0,4064.494472,654.851256,0.196977,0.138210,0.161115
303,0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf,0.607049,337.0,227.0,3293.125263,636.657458,0.314142,0.092193,0.193329
230,0x0aae946810566ff74780fdd8fbc961202acae845,0.514088,503.0,451.0,3849.214000,545.401000,0.285716,0.128804,0.141692
214,0x721935aad682439ad089debd28a59bbcd3f8fcf7,0.491956,405.0,386.0,3319.985500,492.814500,0.245102,0.086645,0.148439


## 4 · Train vs Test metrics for selected wallets

Re-run `compute_wallet_metrics` on the **test slice** restricted to the selected wallets,
then merge training and test metrics side-by-side for comparison.

In [ ]:
best_wallet_set = set(filtered_wallets["wallet"])

# ── test metrics for selected wallets only ────────────────────────────────────
df_test_best = df[~df["is_train"] & df["wallet"].isin(best_wallet_set)]
wallet_vol_test, _ = compute_wallet_metrics(df_test_best)

# ── merge train & test side-by-side ──────────────────────────────────────────
METRIC_COLS = ["wallet", "pnl_volatility", "num_buckets", "num_markets",
               "total_notional", "total_pnl", "return",
               "top5_pnl_pct", "top_market_pnl_pct"]

comparison = (
    filtered_wallets[METRIC_COLS]
    .merge(
        wallet_vol_test[METRIC_COLS],
        on="wallet",
        how="left",
        suffixes=("_train", "_test"),
    )
    .sort_values("total_pnl_train", ascending=False)
    .reset_index(drop=True)
)

print(f"Wallets with test data: {comparison['total_pnl_test'].notna().sum()} / {len(comparison)}")
comparison

Wallets with test data: 22 / 24


,wallet,pnl_volatility_train,num_buckets_train,num_markets_train,total_notional_train,total_pnl_train,return_train,top5_pnl_pct_train,top_market_pnl_pct_train,pnl_volatility_test,num_buckets_test,num_markets_test,total_notional_test,total_pnl_test,return_test,top5_pnl_pct_test,top_market_pnl_pct_test
0,0xcf0aca0d7a395202aec661c3666be9cc098e320a,0.727513,586.0,245.0,39565.522536,5946.969456,0.150307,0.265758,0.078130,3.444729,227.0,79.0,32819.845650,4350.644350,0.132561,0.473247,0.200529
1,0x02319ffcf7214a5ca8e972d409320c935ea545d0,0.746335,421.0,123.0,29709.337690,3276.802310,0.110295,0.187134,0.151404,1.668960,199.0,81.0,26679.144620,1313.855380,0.049247,0.510254,0.117635
2,0x29f8da02e78a87b2d9a844124ff570d5cee34d7b,0.664580,232.0,84.0,5816.015926,1213.979017,0.208730,0.353252,0.192214,NaN,68.0,24.0,1635.701560,-42.781560,-0.026155,NaN,NaN
3,0x2911e7b42a848d0a539ba88ac4a4e7d49480fed6,0.543717,114.0,71.0,8498.980125,1043.505849,0.122780,0.207839,0.073912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0xcceb22d524e186153cffe79f13c0aeb75889f030,0.768612,613.0,364.0,15936.981130,929.917720,0.058350,0.384273,0.196102,NaN,186.0,145.0,6315.565720,-45.614550,-0.007223,NaN,NaN
5,0x8f3ad9fea07b12c950039a668425083ea3d89bdb,0.607948,335.0,315.0,2956.189700,685.410300,0.231856,0.358967,0.202973,NaN,117.0,111.0,1673.550000,-62.250000,-0.037196,NaN,NaN
6,0x766f37ff68fb7e85b122ceb8ab3b1af67d6a9e74,0.381731,135.0,21.0,4064.494472,654.851256,0.161115,0.196977,0.138210,6.530655,34.0,23.0,2286.051069,707.659605,0.309555,0.899077,0.657007
7,0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf,0.607049,337.0,227.0,3293.125263,636.657458,0.193329,0.314142,0.092193,1.666261,171.0,115.0,2468.628924,96.924715,0.039263,1.478390,0.507309
8,0x0aae946810566ff74780fdd8fbc961202acae845,0.514088,503.0,451.0,3849.214000,545.401000,0.141692,0.285716,0.128804,NaN,32.0,31.0,319.344200,-52.974200,-0.165884,NaN,NaN
9,0x721935aad682439ad089debd28a59bbcd3f8fcf7,0.491956,405.0,386.0,3319.985500,492.814500,0.148439,0.245102,0.086645,NaN,53.0,50.0,557.520000,-69.520000,-0.124695,NaN,NaN


In [202]:
comparison[["wallet", "total_pnl_train", "total_pnl_test", "pnl_volatility_train", "pnl_volatility_test", "return_train", "return_test"]].head(20)

,wallet,total_pnl_train,total_pnl_test,pnl_volatility_train,pnl_volatility_test,return_train,return_test
0,0xcf0aca0d7a395202aec661c3666be9cc098e320a,5946.969456,4350.644350,0.727513,3.444729,0.150307,0.132561
1,0x02319ffcf7214a5ca8e972d409320c935ea545d0,3276.802310,1313.855380,0.746335,1.668960,0.110295,0.049247
2,0x29f8da02e78a87b2d9a844124ff570d5cee34d7b,1213.979017,-42.781560,0.664580,NaN,0.208730,-0.026155
3,0x2911e7b42a848d0a539ba88ac4a4e7d49480fed6,1043.505849,NaN,0.543717,NaN,0.122780,NaN
4,0xcceb22d524e186153cffe79f13c0aeb75889f030,929.917720,-45.614550,0.768612,NaN,0.058350,-0.007223
5,0x8f3ad9fea07b12c950039a668425083ea3d89bdb,685.410300,-62.250000,0.607948,NaN,0.231856,-0.037196
6,0x766f37ff68fb7e85b122ceb8ab3b1af67d6a9e74,654.851256,707.659605,0.381731,6.530655,0.161115,0.309555
7,0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf,636.657458,96.924715,0.607049,1.666261,0.193329,0.039263
8,0x0aae946810566ff74780fdd8fbc961202acae845,545.401000,-52.974200,0.514088,NaN,0.141692,-0.165884
9,0x721935aad682439ad089debd28a59bbcd3f8fcf7,492.814500,-69.520000,0.491956,NaN,0.148439,-0.124695


In [ ]:
# comparison['wallet'].tolist()

['0xcf0aca0d7a395202aec661c3666be9cc098e320a',
 '0x02319ffcf7214a5ca8e972d409320c935ea545d0',
 '0x29f8da02e78a87b2d9a844124ff570d5cee34d7b',
 '0x2911e7b42a848d0a539ba88ac4a4e7d49480fed6',
 '0xcceb22d524e186153cffe79f13c0aeb75889f030',
 '0x8f3ad9fea07b12c950039a668425083ea3d89bdb',
 '0x766f37ff68fb7e85b122ceb8ab3b1af67d6a9e74',
 '0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf',
 '0x0aae946810566ff74780fdd8fbc961202acae845',
 '0x721935aad682439ad089debd28a59bbcd3f8fcf7',
 '0xaebcb6fa25b8cffa793d9f48e692fb1ed017bb17',
 '0x41ae78e9bb9f4cf8de1bf619f8b7d9e08957bdfc',
 '0x5f963fb7e95eff26c5868d40af8e838f363092e0',
 '0x711d19594b84eebe94cdf67440c5aa53cc5efe4b',
 '0xe315130229dcf148e87555a6722baf8770dd3911',
 '0x07ebf88cdef21e0aa68e93b21d4002e2a6ea4b4c',
 '0xffcffeec9fe81542525a45d9e119a77dd7afec0e',
 '0x8c80b446e95051c41286a77edcbfa93c9eed6422',
 '0x0f54bf17e6b177d689b66cf55eb8eee91bf093d5',
 '0x399c48e739ff88dfa153d8d9bc220b0d69b62982',
 '0xe51abbc214c832cf44463db0f5f3ed421c742665',
 '0xf4bbc20d5

In [194]:
import plotly.graph_objects as go

# shorten wallet addresses for readability
comparison["wallet_short"] = comparison["wallet"].str[:6] + "..." + comparison["wallet"].str[-4:]

fig = go.Figure([
    go.Bar(
        name="Train PnL",
        x=comparison["wallet_short"],
        y=comparison["total_pnl_train"],
        marker_color="steelblue",
    ),
    go.Bar(
        name="Test PnL",
        x=comparison["wallet_short"],
        y=comparison["total_pnl_test"],
        marker_color="darkorange",
    ),
])
fig.update_layout(
    barmode="group",
    title="Train vs Test PnL per wallet",
    xaxis_title="Wallet",
    yaxis_title="PnL (USDC)",
    xaxis_tickangle=-45,
    legend_title="Period",
)
fig.show(renderer="browser")

In [195]:
fig = go.Figure([
    go.Bar(
        name="Train return",
        x=comparison["wallet_short"],
        y=comparison["return_train"],
        marker_color="steelblue",
    ),
    go.Bar(
        name="Test return",
        x=comparison["wallet_short"],
        y=comparison["return_test"],
        marker_color="darkorange",
    ),
])
fig.update_layout(
    barmode="group",
    title="Train vs Test return (PnL / notional) per wallet",
    xaxis_title="Wallet",
    yaxis_title="Return",
    xaxis_tickangle=-45,
    legend_title="Period",
)
fig.show(renderer="browser")

## 5 · Full-dataset PnL for best wallets

Build timestamp buckets over **all** data (training + test) for the selected wallets,
then plot cumulative PnL with a vertical line marking the train/test split.

In [203]:
import plotly.express as px

# ── full-dataset buckets (train + test) for best wallets ─────────────────────
df_best = df[df["wallet"].isin(best_wallet_set)].copy()
df_best["dt_floored"] = df_best["dt"].dt.floor("5min")
buckets_full = (
    df_best.groupby(["wallet", "dt_floored", "condition_id"], sort=False)
    .agg(notional=("notional", "sum"), pnl=("pnl", "sum"))
    .reset_index()
)
buckets_full = buckets_full[buckets_full["notional"] > 0].copy()

split_line = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

print(f"Full-dataset buckets: {len(buckets_full):,}  across {buckets_full['wallet'].nunique():,} wallets.")
print(f"Train/test split at:  {split_line.date()}")

Full-dataset buckets: 12,844  across 24 wallets.
Train/test split at:  2026-02-02


In [204]:
# ── per-wallet cumulative PnL (top-20 by training PnL, full dataset) ──────────
top_wallets_plot = filtered_wallets.head(20)["wallet"].tolist()

cumulative_pnl = (
    buckets_full[buckets_full["wallet"].isin(top_wallets_plot)]
    .sort_values(["wallet", "dt_floored"])
    .copy()
)
cumulative_pnl["cumulative_pnl"] = cumulative_pnl.groupby("wallet")["pnl"].cumsum()

fig = px.line(
    cumulative_pnl,
    x="dt_floored",
    y="cumulative_pnl",
    color="wallet",
    title="Cumulative PnL Over Time by Wallet (train + test)",
    labels={"dt_floored": "Time", "cumulative_pnl": "Cumulative PnL (USDC)", "wallet": "Wallet"},
)
fig.add_vline(
    x=split_line.timestamp() * 1000,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Train / Test split ({END_DATE_TRAIN})",
    annotation_position="top left",
)
fig.show(renderer="browser")

In [198]:
# ── combined cumulative PnL across all best wallets ───────────────────────────
cumulative_pnl_all = (
    buckets_full[buckets_full["wallet"].isin(best_wallet_set)]
    .sort_values("dt_floored")
    .copy()
)
cumulative_pnl_all["cumulative_pnl"] = cumulative_pnl_all["pnl"].cumsum()

fig = px.line(
    cumulative_pnl_all,
    x="dt_floored",
    y="cumulative_pnl",
    title="Cumulative PnL Over Time (All Best Wallets, train + test)",
    labels={"dt_floored": "Time", "cumulative_pnl": "Cumulative PnL (USDC)"},
)
fig.add_vline(
    x=split_line.timestamp() * 1000,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Train / Test split ({END_DATE_TRAIN})",
    annotation_position="top left",
)
fig.show(renderer="browser")

In [199]:
print(f"len(df_best): {len(df_best):,}")
df_best[['dt', 'title', 'pseudonym', 'condition_id', 'wallet', 'size', 'price', 'pnl', 'is_train']].sort_values('dt').head(10)

len(df_best): 14,089


,dt,title,pseudonym,condition_id,wallet,size,price,pnl,is_train
3459903,2025-10-12 02:18:19+00:00,Will Let God Sort Em Out (Clipse Pusha T Malic...,Energetic-Gasp,0xa14fa7cd7ec5c4472528335369eabdf92e83a1968e8c...,0xcceb22d524e186153cffe79f13c0aeb75889f030,50.00,0.850,-7.50000,True
4505174,2025-10-22 12:31:32+00:00,Will Tesla deliver between 450000 and 475000 v...,Energetic-Gasp,0xd298f1661f93eac60fe29b560b666cb1af92da925182...,0xcceb22d524e186153cffe79f13c0aeb75889f030,100.00,0.130,13.00000,True
1191468,2025-10-22 16:46:48+00:00,Will Luther (Kendrick Lamar and SZA) win Song ...,Energetic-Gasp,0x378f76341e555b6d54b89cc527afde5f8c293025cf6f...,0xcceb22d524e186153cffe79f13c0aeb75889f030,10.00,0.230,2.30000,True
4066877,2025-10-26 02:04:15+00:00,Will Timeless (The Weeknd feat. Playboi Carti)...,Energetic-Gasp,0xbe72aa7e325b9ebb7c1300c79a3fa25520b90845333c...,0xcceb22d524e186153cffe79f13c0aeb75889f030,20.00,0.944,-1.12000,True
3372157,2025-10-28 03:00:07+00:00,Will GNX (Kendrick Lamar) win Best Rap Album a...,Energetic-Gasp,0x9d358ad28d4d4cddc736b11f2b9faebb7ced21825828...,0xcceb22d524e186153cffe79f13c0aeb75889f030,23.80,0.380,9.04400,True
3459904,2025-10-28 09:24:23+00:00,Will Let God Sort Em Out (Clipse Pusha T Malic...,Energetic-Gasp,0xa14fa7cd7ec5c4472528335369eabdf92e83a1968e8c...,0xcceb22d524e186153cffe79f13c0aeb75889f030,50.00,0.800,-10.00000,True
894265,2025-10-31 15:10:04+00:00,Will MrBeast's next video get between 80 and 9...,Brief-Adverb,0x2920e449f20bd3288dd708132f5ff19f34b0fb855d52...,0x0f54bf17e6b177d689b66cf55eb8eee91bf093d5,99.99,0.015,-1.49985,True
4573422,2025-11-03 18:40:59+00:00,Will Abracadabra (Lady Gaga) win Song of the Y...,Energetic-Gasp,0xd55b7e07b66b1d6702aa8f8c0c761a4f2838b55fa313...,0xcceb22d524e186153cffe79f13c0aeb75889f030,4.80,0.160,0.76800,True
3459905,2025-11-05 06:52:15+00:00,Will Let God Sort Em Out (Clipse Pusha T Malic...,Energetic-Gasp,0xa14fa7cd7ec5c4472528335369eabdf92e83a1968e8c...,0xcceb22d524e186153cffe79f13c0aeb75889f030,60.00,0.660,-20.40000,True
1099557,2025-11-06 15:35:23+00:00,Will APT (Rosé and Bruno Mars) win Song of the...,Energetic-Gasp,0x32821c03c1a785af6bea72291b578e722f02f7da80e2...,0xcceb22d524e186153cffe79f13c0aeb75889f030,10.00,0.930,-0.70000,True


In [200]:
df_best.groupby("condition_id").agg(
    title          = ("title",          "first"),
    total_pnl      = ("pnl",            "sum"),
    num_trades     = ("transactionHash", "count"),
    total_notional = ("notional",        "sum"),
    wallet_count   = ("wallet",          "count"),
).sort_values("total_pnl", ascending=False).head(30)

,title,total_pnl,num_trades,total_notional,wallet_count
condition_id,,,,,
0x8db6193b4622d0517aade7493b762488ca50a9ad86d89edd4fd14a61011fae79,LoL: SK Gaming vs GIANTX (BO1) - LEC Versus Re...,874.131800,9,630.868200,9
0x987ae8ab6025fb220a5acf297aa367d14931c85d14484ed730e00d1fd7f605c7,Dota 2: OG vs Team Spirit (BO1) - BLAST Slam G...,718.922084,14,1262.193299,14
0x7f775504ea5e50064ffcbddfdcd65894ba2378b0b9b129729a28af5142da254e,LoL: Team Heretics vs Karmine Corp (BO1) - LEC...,622.318578,8,380.293498,8
0xfa549195ef0c840f581db765bce9b85aa3e866f86be463cf79d0256c2bcab8e4,Will the highest temperature in Toronto be -1°...,496.120900,43,619.929100,43
0x2e287530822cb932510bc680e4f8a01558e37804e4cb94c2f0a4bc0c7aca9598,LoL: Team Heretics vs Movistar KOI (BO1) - LEC...,488.422500,8,811.577500,8
0x01b8c21fdeacf1fc8b705cdf3ebc9f3d48763478beb84ff477280b22942190ca,Will Keri Russell win Best Actress in a Drama ...,464.937448,2,408.121552,2
0x2dc1b43e487916555ac92a3abde3a9de015bb6ac0d6e53cea626559b8b7faf0d,LoL: Natus Vincere vs Fnatic (BO1),464.636500,7,186.213500,7
0x275204a8663ac92b410f006e6b187d949528ff31d53d1b1c9c6724724053a598,Dota 2: Aurora vs AVULUS - Game 1 Winner,463.744600,4,736.255400,4
0x95c4b7edf2e74482cf05855c242cb7818662bd947002ef3fcaeb4929a04694a1,Dota 2: Team Liquid vs MOUZ (BO1) - BLAST Slam...,373.314302,13,609.225698,13
